# 先上线：最朴素的部署方案

## 朴素的部署方案

项目在本地运行时：

- 前端通过 `npm run dev` 监听 3000 端口。
- 后端通过 `uv run fastapi dev` 监听 8000 端口。
- 浏览器访问前端，前端 JavaScript 再通过 `fetch` 请求后端。

服务器也是一台电脑，所以先采用最直接的方案：

~~~text
浏览器
  ├── http://服务器IP          → Nginx:80 → 静态前端
  └── http://服务器IP:8000     → Uvicorn  → FastAPI 后端
~~~

前端不再用开发服务器监听 3000 端口，而是构建成静态文件，继续由 Nginx 通过 80 端口提供。

后端暂时直接通过 Uvicorn 在 8000 端口提供服务。

## 我们已经知道的

### 1. 前端部署

前端代码推送到服务器后，安装依赖、构建静态文件，再交给 Nginx 提供。

### 2. Python 环境

Ubuntu 通常自带 Python。后端运行前需要确认 Python 已安装且版本足够。

### 3. 虚拟环境与依赖

项目使用 `uv` 管理 Python 依赖，在家目录安装：

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh

source $HOME/.local/bin/env #刷新环境变量
```

执行：

~~~bash
uv sync --default-index https://pypi.tuna.tsinghua.edu.cn/simple
~~~

### 4. 代码和数据库

`main.py`、`storage.py` 等代码通过 Git 和 GitHub 同步。`history.db` 不进 Git，但后端启动时会自动创建。

### 5. 启动后端

本地一直使用 `uv run fastapi dev`，背后仍然由 Uvicorn 监听 8000 端口并提供 API。

## 服务器部署会遇到的挑战

### 挑战 1：缺少前端配置文件

`.env.local` 不进 Git，服务器拉取代码后没有前端配置。而且 `.local` 表示本机私有，线上生产构建应使用另一种配置文件。

### 挑战 2：后端配置不匹配

后端 CORS 目前写死了本地地址：

~~~python
allow_origins=["http://localhost:3000"]
~~~

线上用户不会从 `localhost:3000` 访问。这种随环境变化的值不应该写死在代码里，也应该放入配置文件。

### 挑战 3：`fastapi dev` 只适合本地开发

`fastapi dev` 默认只监听本机地址，外部用户无法通过互联网连接。线上应使用 `fastapi run`。

### 挑战 4：终端关闭后端就停止

后端目前是终端中的前台进程。SSH 窗口一关，服务就停止。

### 挑战 5：8000 端口没有放行

云平台安全组或防火墙通常没有开放 8000 端口，需要手动放行。

五条挑战归拢后是四件事：

1. 在本地处理前后端配置文件。
2. 把 `uv run fastapi dev` 换成 `uv run fastapi run`。
3. 在云平台开放 8000 端口。
4. 最后让后端在后台持续运行。

## 配置文件

通常使用“名称=值”的形式：

~~~dotenv
名称a=xxx
名称b=123
~~~

## 项目中已有的两处地址配置

前端根目录中的 `.env.local`：

~~~dotenv
NEXT_PUBLIC_API_BASE_URL=http://localhost:8000
~~~

前端通过 `NEXT_PUBLIC_API_BASE_URL` 找到后端地址。

后端 `main.py` 中则写死了：

~~~python
allow_origins=["http://localhost:3000"]
~~~

它表示允许哪个前端源跨源访问后端。这次要把它也抽到配置文件中。

## 配置文件的命名和读取

配置文件通常叫 `.env`，但具体名称由框架约定。

Next.js 会识别：

- `.env.local`：本机私有配置。
- `.env.development.local`。
- `.env.development`。
- `.env.production`：生产构建时使用。
- `.env`：各环境通用，优先级较低。
- `。env.example`:样板，需要进 git

`npm run dev` 时，常见查找优先级为：

~~~text
.env.development.local
→ .env.local
→ .env.development
→ .env
~~~

`npm run build` 时，中间的 `development` 会换成 `production`。

Python 后端使用 `python-dotenv` 读取 `.env`。前端配置放在项目根目录，后端配置放在 `backend/` 目录。以点开头的文件在 Linux 中是隐藏文件。

## 动手处理后端配置

先在 `backend/` 中创建 `.env`：

~~~dotenv
# backend/.env
ALLOWED_ORIGINS=http://localhost:3000
~~~

确认 `.gitignore` 中有：

~~~gitignore
.env*
~~~

## 让 Python 读取配置

修改 `backend/main.py`。顶部加入：

~~~python
import os
from dotenv import load_dotenv

load_dotenv()
ALLOWED_ORIGINS = os.getenv("ALLOWED_ORIGINS").split(",")
~~~

把 CORS 中写死的地址替换掉：

~~~python
app.add_middleware(
    CORSMiddleware,
    allow_origins=ALLOWED_ORIGINS,
    allow_credentials=True,
    allow_methods=["GET", "POST"],
)
~~~

`python-dotenv` 只负责把 `.env` 中的内容读入环境变量。`os.getenv()` 再从环境变量中读取指定值。

## 安装 `python-dotenv` 

`dotenv` 不是 Python 标准库，需要安装，并重新生成依赖清单：

~~~bash
uv add python-dotenv
~~~

本地启动前后端进行验证：

~~~bash
# 后端
uv run fastapi dev

# 前端，另开终端
npm run dev
~~~

文字实验室仍然能正常使用，说明后端已经成功从配置中读取 CORS 地址。

## 创建前后端配置示例

在项目根目录创建 `.env.example`：

~~~dotenv
NEXT_PUBLIC_API_BASE_URL=http(s)://[ip]:[port]
~~~

在 `backend/` 中创建 `.env.example`：

~~~dotenv
ALLOWED_ORIGINS=http(s)://[ip]:[port]
~~~

键名必须完整，值应提供可用示例或清晰指引。这样别人复制文件后只需修改值，不用猜键名。

## 让示例配置进入 Git

`.gitignore` 中的 `.env*` 也会匹配 `.env.example`，因此需要在它下面增加一个例外：

~~~gitignore
.env*
!.env.example
~~~

这表示忽略真实配置，但允许所有目录中的 `.env.example` 被 Git 追踪。

完成后提交本地修改：

~~~bash
git add .
git commit -m "CORS 名单改为从配置读取;补 .env.example"
git push
~~~

本次提交包括：

- 读取配置的 `main.py`。
- 增加 `python-dotenv` 的 `requirements.txt`。
- 前后端两份 `.env.example`。
- 更新后的 `.gitignore`。

## 服务器部署：拉取代码

SSH 登录服务器，然后拉取最新代码：

~~~bash
ssh 用户名@你的服务器IP

cd ~/zero-to-tech
git pull
~~~

检查后端目录：

~~~bash
ls backend/
~~~

## 准备 Python 环境

先检查服务器上的 Python：

~~~bash
python3 --version
which python3
~~~

两条命令回答不同问题：

- `python3 --version`：是否安装，版本是否满足依赖要求。
- `which python3`：当前使用的是哪一个 Python。

服务器可能同时存在多个 Python，`python` 和 `python3` 也可能指向不同位置。

Python 3.10 或更高版本通常可以满足本项目。如果版本过低或未安装，需要先安装合适的 Python。

Node.js 在前端部署时已经安装过，不放心可以检查：

~~~bash
node -v
~~~

## 在服务器安装 uv

项目使用 `uv` 管理 Python 依赖，在家目录安装：

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh

source $HOME/.local/bin/env #刷新环境变量
```

执行（使用镜像）：

~~~bash
uv sync --default-index https://pypi.tuna.tsinghua.edu.cn/simple
~~~

## 在服务器写前端配置

进入项目根目录，从示例复制生产配置：

~~~bash
cd ~/zero-to-tech
ls -a #查看隐藏文件
cp .env.example .env.production
vim .env.production
~~~

写入：

~~~dotenv
NEXT_PUBLIC_API_BASE_URL=http://服务器IP:8000
~~~

前端生产构建时会读取 `.env.production`，生成的静态 JavaScript 将通过这个地址请求后端。

## 在服务器写后端配置

进入后端目录：

~~~bash
cd ~/zero-to-tech/backend
ls -a #查看隐藏文件
cp .env.example .env
vim .env
~~~

写入：

~~~dotenv
ALLOWED_ORIGINS=http://服务器IP
~~~

这里不要写 `:3000`，也不需要写 `:80`。

| 配置 | 本地 | 线上 |
| --- | --- | --- |
| `NEXT_PUBLIC_API_BASE_URL` | `http://localhost:8000` | `http://服务器IP:8000` |
| `ALLOWED_ORIGINS` | `http://localhost:3000` | `http://服务器IP` |

`ALLOWED_ORIGINS` 登记的是前端页面所在的源。本地前端运行在 3000 端口；线上前端由 Nginx 在 80 端口提供，而 80 是 HTTP 默认端口，浏览器发出的 Origin 中不会写出它。

## 跑起来：构建前端

进入项目根目录：

~~~bash
cd ~/zero-to-tech
npm install
npm run build
~~~

Next.js 读取 `.env.production`，构建生成静态前端文件。Nginx 继续通过 80 端口提供这些静态资源。

## 跑起来：启动后端

进入后端目录并启动生产服务：

~~~bash
cd ~/zero-to-tech/backend

uv run fastapi run
~~~

背后仍然是 Uvicorn，但 `run` 与 `dev` 的用途不同：

| `fastapi dev` | `fastapi run` |
| --- | --- |
| 修改代码后自动重启 | 不自动重启 |
| 默认监听 `127.0.0.1` | 默认监听 `0.0.0.0` |
| 适合本机开发 | 适合服务器运行 |

- `127.0.0.1` 只接受服务器内部连接。
- `0.0.0.0` 监听所有网卡，外部用户也能连接。

## 先在服务器本地验证

在后端运行期间，再打开一个终端并重新 SSH 登录。

先测试后端接口：

~~~bash
curl localhost:8000/api/profile
~~~

能看到 JSON，说明后端已经启动，并且正在监听 8000 端口。

再检查数据库：

~~~bash
ls ~/zero-to-tech/backend/
~~~

如果出现 `history.db`，说明应用启动时已成功初始化 SQLite 数据库。

公网访问之前，应该先确认服务器本机访问正常。这样公网访问失败时，排查范围可以集中在端口和网络，而不是应用代码。

## 开通 8000 端口

前端通过 80 端口访问，后端暂时通过 8000 端口访问。80 端口此前已经开放，但云平台安全组或防火墙通常不会默认开放 8000。

在云平台控制台中添加入站规则，放行 TCP 8000 端口。操作方法与此前开放 80 端口相同。

开放后，在本地电脑访问：

~~~text
http://服务器IP:8000/api/profile
~~~

如果浏览器得到 JSON，说明 8000 端口已经能从公网访问。

## 完整线上测试

打开：

~~~text
http://服务器IP
~~~

依次验证：

1. 打开文字实验室，分析一句文字。它测试前端能否请求线上后端。
2. 打开历史记录，确认刚才的内容存在。它测试 SQLite 是否正常工作。
3. 换一个浏览器或打开无痕窗口，分析另一句话，再查看历史。
4. 两个浏览器应该只能看到各自的记录。它测试 6.6 完成的会话机制。

如果全部正常，zero-to-tech 已经可以通过电脑或手机浏览器从互联网访问。

## 服务端的后台运行

此时后端仍然是前台进程，寄生在当前 SSH 会话中。关闭 SSH 窗口后：

- Nginx 提供的静态前端仍然存在。
- Uvicorn 后端停止，分析按钮会请求失败。

我们希望后端不依赖 SSH 会话。重新登录服务器后执行：

~~~bash
cd ~/zero-to-tech/backend
nohup uv run fastapi run main.py > backend.log 2>&1 &
~~~

这条命令有四个关键部分：

- 末尾的 `&`：把命令放到后台运行，终端不用一直等待。
- 开头的 `nohup`：no hang up，让进程不因 SSH 断开而被终止。只有 `&` 还不够。
- `> backend.log`：把正常输出重定向到日志文件。
- `2>&1`：让错误输出 2 也跟随正常输出 1，写入同一个日志文件。

执行后终端会显示一个数字，它是进程号 PID。

## 查看日志和停止服务

这套朴素方案需要手动管理后端,在 backend 目录：

实时查看日志。

~~~bash
tail -f backend.log
~~~

查找 FastAPI 进程及其 PID。

~~~bash
ps aux | grep fastapi
~~~

停止对应进程。

~~~bash
kill 进程号
~~~

这些命令也将写入项目 README，方便以后部署和维护。

## 回顾并写入 `README.md`

README 至少要说明：

- 项目用途和技术栈。
- 本地如何启动前后端。
- 如何部署到服务器。
- 前后端配置文件的键名和示例。
- 如何启动、查看日志和停止后端。
- 需要放行哪些端口。
- 如何验证分析、历史记录和会话。

# 文字实验室

一个中文文本分析小工具：输入一段话，给出情感倾向评分和全文拼音，
并把每次分析的结果存下来，各人只看得到自己的那一份。

零到全栈课程的贯穿项目。

## 技术栈

- 前端：Next.js（静态导出）＋ React
- 后端：FastAPI ＋ uvicorn
- 分析：snownlp（情感）、pypinyin（注音）
- 存储：SQLite
- 线上：Nginx

## 本地跑起来

需要：Node.js 18+、Python 3.10+

**后端**

```bash
cd backend
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
cp .env.example .env          # 按下面「配置说明」填好
fastapi dev                   # → http://localhost:8000
```

**前端**（另开一个终端）

```bash
npm install
cp .env.example .env.local    # 按下面「配置说明」填好
npm run dev                   # → http://localhost:3000
```

## 部署到服务器

前提：服务器上已装好 Python 3.10+、Node.js 18+ 和 Nginx，
且 Nginx 的站点根目录已指向本项目的 `out/`、监听 80 端口。

**1. 拉取代码**

```bash
cd ~/zero-to-tech
git pull
```

**2. 前端：装依赖、写配置、构建**

```bash
npm install
cp .env.example .env.production   # 按下面「配置说明」填好
npm run build                     # 产物进 out/，由 Nginx 提供服务
```

**3. 后端：建环境、装依赖、写配置**

```bash
cd backend
python3 -m venv --prompt=zero-to-tech .venv   # 首次部署才需要
source .venv/bin/activate
pip install -r requirements.txt
cp .env.example .env              # 按下面「配置说明」填好
```

**4. 后端：在后台跑起来**

```bash
nohup .venv/bin/fastapi run > backend.log 2>&1 &
```

`fastapi run` 是生产模式，监听 `0.0.0.0:8000`；`nohup ... &` 让它在
SSH 断开后继续运行，日志写进 `backend.log`。

查看日志、停止服务：

```bash
tail -f backend.log           # 看日志
ps aux | grep fastapi         # 找到进程号
kill 进程号                    # 停掉
```

**5. 放行 8000 端口**

去云平台控制台的安全组 / 防火墙，放行 8000 端口（80 端口应该已经放行）。

**6. 验证**

浏览器访问 `http://服务器IP`，打开文字实验室做一次分析，再看历史记录。
换一个浏览器（或无痕窗口）再试一次，两边的历史记录应该是互相看不到的。

## 配置说明

配置文件不进 Git，请照着 `.env.example` 自己建一份。

**前端**：开发用 `.env.local`，生产构建用 `.env.production`

| 键 | 说明 | 本地 | 线上 |
| --- | --- | --- | --- |
| `NEXT_PUBLIC_API_BASE_URL` | 后端接口地址 | `http://localhost:8000` | `http://服务器IP:8000` |

**后端**：`backend/.env`

| 键 | 说明 | 本地 | 线上 |
| --- | --- | --- | --- |
| `ALLOWED_ORIGINS` | 允许跨源访问的前端地址，多个用逗号隔开 | `http://localhost:3000` | `http://服务器IP`（不带端口） |


## 思考：现在的部署方式好不好？

站点已经上线，但这套最朴素的方案有三个明显问题。

### 第一：后端服务脆弱

`nohup` 只解决 SSH 断开后继续运行，并没有解决：

- 服务器重启后不会自动启动。
- 进程崩溃后不会自动恢复。
- 更新代码后停止和重启都要手动查 PID。
- 查看服务状态不方便。
- 日志位置需要人工记忆。

### 第二：8000 端口直接暴露公网

- 8000 端口没有 Nginx 的访问日志。
- 请求内容没有 HTTPS 加密。
- 没有限流和转发规则。
- 多开放一个端口，就多一份风险。

### 第三：前后端仍然跨源

浏览器 CORS 对跨源请求有很多限制。以后增加域名和 HTTPS 时，跨源问题仍会继续出现。更好的办法是在部署层让前后端同源。

## 下一节要解决什么？

下一节将逐一替换这套朴素方案：

- 用 **systemd** 代替 `nohup`：实现开机自启、崩溃重启、统一查看状态和日志。
- 用 **Nginx 反向代理**把 `/api/` 转发到后端：8000 端口不再对公网开放。
- 前端和后端都通过 80 端口访问，自然变成同源，配置也会更简单。